对数据进行初步清洗

In [ ]:
import json
import re


def remove_duplicate_content(user_content, assistant_content):
    """
    精确删除assistant回复中开头重复的user内容

    Args:
        user_content: 用户投诉内容
        assistant_content: 助理回复内容

    Returns:
        str: 清理后的回复内容
    """
    if not user_content or not assistant_content:
        return assistant_content

    user_clean = user_content.strip()
    assistant_clean = assistant_content.strip()

    # 提取user内容的主要部分（去掉最后的"注："部分）
    user_main_content = re.split(r"注：|备注：", user_clean)[0].strip()

    # 检查assistant是否以user主要内容开头
    if assistant_clean.startswith(user_main_content):
        remaining = assistant_clean[len(user_main_content) :].lstrip("，。.,;；")
        return remaining if remaining else assistant_clean

    return assistant_clean


def clean_structured_complaint_data(input_file, output_file):
    """
    专门处理结构化投诉数据的清理和转换
    """
    # 读取原始数据
    with open(input_file, "r", encoding="utf-8") as f:
        original_data = json.load(f)

    # 专业系统提示词
    system_prompt = """你是一名专业的政府投诉处理AI助手，专门负责生成标准化的投诉处理回复。

你的职责包括：
1. 准确理解市民的投诉内容
2. 严格按照政府工作流程和法律法规进行回复
3. 使用正式、规范的公务语言

请根据市民投诉内容，生成符合上述要求的专业回复。"""

    cleaned_data = []
    processed_count = 0
    cleaned_count = 0

    for item in original_data:
        try:
            # 提取字段
            instruction = item.get("instruction", "")
            user_input = item.get("input", "")
            original_output = item.get("output", "")

            # 将input内容合并到instruction中，因为原数据中instruction都是空的
            # 所以直接使用input作为instruction
            if user_input:
                user_query = user_input
            elif instruction:
                user_query = instruction
            else:
                user_query = ""

            # 清理重复内容
            cleaned_output = remove_duplicate_content(user_query, original_output)

            if cleaned_output != original_output:
                cleaned_count += 1
                print(f"✅ 已清理重复内容: {cleaned_output[:100]}...")

            # 构建标准格式 - 使用instruction/input/output/system格式
            training_example = {
                "instruction": user_query.strip(),
                "input": "",  # 用户输入（选填），这里设为空
                "output": cleaned_output.strip(),
                "system": system_prompt,
            }

            cleaned_data.append(training_example)
            processed_count += 1

        except Exception as e:
            print(f"处理数据时出错: {e}")
            continue

    # 保存清理后的数据 - 使用JSON数组格式
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

    print("\n📊 处理完成!")
    print(f"处理数据: {processed_count} 条")
    print(f"清理重复: {cleaned_count} 条")
    print(f"输出文件: {output_file}")


# 使用示例
if __name__ == "__main__":
    # 处理完整数据集
    print("\n🔄 处理完整数据集...")
    input_file = "server/data/train_dataset.json"
    output_file = "server/data/training_data_formatted.json"

    clean_structured_complaint_data(input_file, output_file)


转换结构

In [1]:
import os
import re

# ================= 配置区域 =================
# 将路径修改为你的主文件夹名称
INPUT_FOLDER = './Office_law' 
# ===========================================

class Document:
    def __init__(self, page_content, metadata=None):
        self.page_content = page_content
        self.metadata = metadata or {}

    def __repr__(self):
        # 打印时显示来源、分类和条号
        return f"<Doc source='{self.metadata.get('source')}' category='{self.metadata.get('category')}' id='{self.metadata.get('id')}'> len={len(self.page_content)}"

def clean_noise_lines(text):
    """Step 1: 预清洗 - 去除网页噪音"""
    lines = text.split('\n')
    cleaned_lines = []
    
    noise_keywords = [
        "索引号：", "主题分类：", "文号：", "发布日期：", "公文生成日期：",
        "分享 |", "官方微信", "无障碍浏览", "版权所有", "ICP备",
        "网站地图", "当前位置：", "相关链接", "扫一扫", "PDF文件"
    ]
    
    for line in lines:
        line_strip = line.strip()
        if not line_strip: continue 
        if line_strip.startswith("【") and "】" in line_strip: continue
        if any(kw in line_strip for kw in noise_keywords): continue
        cleaned_lines.append(line)
        
    return "\n".join(cleaned_lines)

def is_chapter_header(line):
    """判断是否为章节标题（如：第一章 总则）"""
    pattern = r"^\s*第[零一二三四五六七八九十百]+[章节]"
    return bool(re.match(pattern, line.strip()))

def split_standard_law(text, filename, category):
    """处理标准法律 (按'第X条'切分)"""
    chunks = []
    source_name = filename.replace('.txt', '')
    article_pattern = r"(^|\n)\s*(第[零一二三四五六七八九十百]+条)"
    splits = re.split(article_pattern, text)
    
    i = 0
    while i < len(splits):
        part = splits[i]
        if re.match(r"\s*第[零一二三四五六七八九十百]+条", part.strip()):
            current_article_title = part.strip()
            if i + 1 < len(splits):
                raw_content = splits[i+1]
                # 行级清洗：剔除夹在中间的章节标题
                content_lines = raw_content.split('\n')
                valid_lines = [line.strip() for line in content_lines if line.strip() and not is_chapter_header(line.strip())]
                final_content = "".join(valid_lines)
                
                if final_content:
                    full_text = f"{current_article_title} {final_content}"
                    chunks.append(Document(
                        page_content=full_text, 
                        metadata={
                            "source": source_name, 
                            "id": current_article_title, 
                            "type": "article",
                            "category": category  # 新增：记录子文件夹名称
                        }
                    ))
            i += 1 
        i += 1
    return chunks

def split_policy_doc(text, filename, category):
    """处理政策文件 (按'一、'切分)"""
    chunks = []
    source_name = filename.replace('.txt', '')
    item_pattern = r"(^[一二三四五六七八九十]+、)"
    splits = re.split(item_pattern, text, flags=re.MULTILINE)
    
    for i in range(1, len(splits), 2):
        if i + 1 < len(splits):
            title = splits[i].strip()
            content = splits[i+1].strip()
            if content:
                full_text = f"{title}{content}"
                chunks.append(Document(
                    page_content=full_text, 
                    metadata={
                        "source": source_name, 
                        "id": title, 
                        "type": "item",
                        "category": category # 新增
                    }
                ))
    return chunks

def process_legal_files():
    all_docs = []
    
    if not os.path.exists(INPUT_FOLDER):
        print(f"❌ 错误：找不到文件夹 {INPUT_FOLDER}")
        return []

    print(f"📂 开始遍历 {INPUT_FOLDER} 及其子文件夹...\n" + "-"*40)

    # === 修改的核心部分：使用 os.walk 递归遍历 ===
    for root, dirs, files in os.walk(INPUT_FOLDER):
        # root: 当前正在遍历的文件夹路径 (例如 ./Office_law/Civil)
        # dirs: 当前文件夹下的子文件夹列表
        # files: 当前文件夹下的文件列表
        
        # 获取当前子文件夹的名称作为分类 (例如 'Civil')
        current_category = os.path.basename(root)
        if current_category == os.path.basename(INPUT_FOLDER):
            current_category = "root" # 如果文件直接在根目录下

        for f in files:
            if f.endswith('.txt'):
                file_path = os.path.join(root, f)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f_obj:
                        raw_text = f_obj.read()
                    
                    # 1. 清洗
                    cleaned_text = clean_noise_lines(raw_text)
                    
                    # 2. 切分 (传入 category 参数)
                    if cleaned_text.count("第一条") > 0 or cleaned_text.count("第1条") > 0:
                        docs = split_standard_law(cleaned_text, f, current_category)
                    else:
                        docs = split_policy_doc(cleaned_text, f, current_category)
                    
                    all_docs.extend(docs)
                    print(f"✅ [{current_category}] {f} -> 切分出 {len(docs)} 条")
                    
                except Exception as e:
                    print(f"❌ 读取错误 {f}: {e}")

    return all_docs

if __name__ == "__main__":
    # 执行处理
    documents = process_legal_files()
    
    print("-" * 40)
    print(f"🎉 全部完成！总共获得 {len(documents)} 个结构化数据块。")
    
    if documents:
        print("\n🔍 数据结构抽查:")
        # 打印第一个结果看看 Metadata 是否包含 category
        print(documents[0])

📂 开始遍历 ./Office_law 及其子文件夹...
----------------------------------------
✅ [法律] 中华人民共和国个人独资企业法.txt -> 切分出 48 条
✅ [法律] 中华人民共和国个人所得税法.txt -> 切分出 24 条
✅ [法律] 中华人民共和国中医药法.txt -> 切分出 63 条
✅ [法律] 中华人民共和国中小企业促进法.txt -> 切分出 61 条
✅ [法律] 中华人民共和国个人信息保护法.txt -> 切分出 75 条
✅ [法律] 中华人民共和国乡镇企业法.txt -> 切分出 43 条
✅ [法律] 中华人民共和国企业国有资产法.txt -> 切分出 77 条
✅ [法律] 中华人民共和国产品质量法.txt -> 切分出 74 条
✅ [法律] 中华人民共和国价格法.txt -> 切分出 48 条
✅ [法律] 中华人民共和国企业破产法.txt -> 切分出 136 条
✅ [法律] 中华人民共和国体育法.txt -> 切分出 122 条
✅ [法律] 中华人民共和国保险法.txt -> 切分出 185 条
✅ [法律] 中华人民共和国全民所有制工业企业法.txt -> 切分出 69 条
✅ [法律] 中华人民共和国保守国家秘密法.txt -> 切分出 65 条
✅ [法律] 中华人民共和国公务员法.txt -> 切分出 113 条
✅ [法律] 中华人民共和国公司法.txt -> 切分出 266 条
✅ [法律] 中华人民共和国公职人员政务处分法.txt -> 切分出 68 条
✅ [法律] 中华人民共和国农业机械化促进法.txt -> 切分出 35 条
✅ [法律] 中华人民共和国农业法.txt -> 切分出 99 条
✅ [法律] 中华人民共和国刑事诉讼法.txt -> 切分出 307 条
✅ [法律] 中华人民共和国刑法.txt -> 切分出 505 条
✅ [法律] 中华人民共和国农民专业合作社法.txt -> 切分出 74 条
✅ [法律] 中华人民共和国劳动法.txt -> 切分出 107 条
✅ [法律] 中华人民共和国劳动合同法.txt -> 切分出 98 条
✅ [法律] 中华人民共和国反不正当竞争法.txt -> 切分出 47 条
✅ [法律] 中华人